# Synergistic Fluorinated Amide Additives: Colab DFT Automation


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT='/content/drive/MyDrive/DFT_Automation'


In [ ]:
!bash src/install_orca.sh


## 1) Create 10 jobs + ORCA input decks


In [ ]:
!python src/job_factory.py


## 2) Copy XYZ templates into each job folder (customize geometries)


In [ ]:
import shutil, pathlib
root=pathlib.Path(PROJECT_ROOT)
for jd in root.glob('Job_*'):
    for name in ['control.xyz','tfa.xyz','ndt.xyz','synergy.xyz']:
        src=pathlib.Path('inputs')/name
        if src.exists(): shutil.copy(src,jd/name)


## 3) Two-stage run: XTB pre-opt then PBEh-3c production


In [ ]:
import subprocess, pathlib
job='Job_01'
wd=pathlib.Path(PROJECT_ROOT)/job
subprocess.run(['orca',str(wd/'preopt_xtb.inp')],cwd=wd)
subprocess.Popen(['orca',str(wd/'job.inp')],cwd=wd)


## 4) Smart monitor (500 s polling, restart on 20-cycle energy stall)


In [ ]:
!python src/monitor.py --workdir "$PROJECT_ROOT/Job_01" --xyz-file "$PROJECT_ROOT/Job_01/ndt.xyz" --poll-seconds 500


## 5) Analysis + markdown draft


In [ ]:
!python src/analysis_XAI.py --jobs-root "$PROJECT_ROOT" --output-dir "$PROJECT_ROOT/analysis"
